# 04 · Fleet rates, customers vs. their fleets, and persistence

This notebook replicates the Rates by model and age, Customers vs. their fleets, and Persistence and outliers paragraphs of the report's Results section.

> Rates by model and age: The fleet runs at 6.2 unplanned tickets per 100 installed machine-months. The three busiest models (LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0) hold 42% of machine-months and 51% of tickets, partly because they are younger. scrubBank 480 runs 2.6 times the fleet age for age. Machines under 4 years run about 8 per 100 vs. 2.5 at 10 years or more, but the 10+ rate overstates the drop with age: 83 of its 164 machines with a ticket are Not Serviceable, and removed machines are missing from the list.
>
> Customers vs. their fleets: 63 customers exceed their fleets beyond chance (1,732 tickets vs. 747 expected), and 1 lies outside the normal spread. The top 10% of customers hold 47% of tickets vs. 39% from their fleets plus chance.
>
> Persistence and outliers: Customers above their fleets in the first half tend to stay above them in the second (rank correlation 0.46 vs. 0.04 when shuffled), and machines with 3 or more first-half tickets had 2.6 times their expected tickets in the second half. 18 customers and 36 machines are consistent outliers. Of the 106 chronic outliers on IPA's list, 93 stay flagged, and no flagged machine stands out as a separate group.

**How it works.** `fleet_exposure.py` joins the unplanned tickets to IPA's installed-base list, counts each machine's installed months, fits each machine's expected tickets from its model and age band (over the full period and within each half), and flags customers and machines. We recompute every count, share, rate, and ratio from its per-machine and per-customer tables (`fleet/fx_machine_flags.csv` and `fleet/fx_customer_rates.csv`), and we take installed months by model and by age band from its per-model and per-age-band tables, because the per-machine table lists only the machines with a ticket. The quantities that need the script's fits or resampling over every fleet machine, including those without a ticket, come from its tables: expected tickets (per machine, per customer, and at the fleet's age-band rates by model), the method-variant flag counts, the negative-binomial q-values, and the two chance benchmarks, which we replay or redraw as cross-checks.

**Re-run:** `verification/fleet_exposure.py` (about 35 s). With `RERUN = True`, it rewrites its 20 `fleet/fx_*.csv` tables and `fleet/fx_summary.json` in a scratch copy of `verification/outputs/`, and the notebook compares each file with the committed one.

**Used as committed, without re-running:** the script also reads `htm_device_exposure.csv`, which `high_ticket_machines.py` writes from the telemetry-database extracts (notebook 02 re-runs it). The script takes docs/05's 113 chronic outliers from it, and the telemetry-span exposure for its interim telemetry-type rates, which these paragraphs do not use; this notebook reads it to identify the 113. The optional extract `fleet/db/fxdb_devices.csv`, which `fleet_db_extract.py` writes from the telemetry database, is not in the committed outputs, so the re-run keeps the telemetry-span basis for telemetry types, as the committed outputs do. `docs/06_fleet_exposure.md` describes the analysis.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'fleet_exposure.py')
R.compare(OUT)

fleet_exposure.py: exit status 0, 46 s
    27/27 checks pass
21 files written: 21 identical.


,file written by the re-run,compared with the committed file
0,fleet/fx_age_rates.csv,identical
1,fleet/fx_country_rates.csv,identical
2,fleet/fx_coverage_rates.csv,identical
3,fleet/fx_customer_lorenz.csv,identical
4,fleet/fx_customer_rates.csv,identical
5,fleet/fx_family_rates.csv,identical
6,fleet/fx_machine_flags.csv,identical
7,fleet/fx_match.csv,identical
8,fleet/fx_model_rates.csv,identical
9,fleet/fx_outlier_customers.csv,identical


## The fleet rate: 6.2 per 100 installed machine-months

The fleet rate is the fleet's unplanned tickets divided by its installed machine-months, times 100. `fleet/fx_customer_rates.csv` has one row per customer with the tickets and installed months of all its machines, including those without a ticket, so we sum it, and we check the ticket total against `fleet/fx_machine_flags.csv`, which has one row per fleet machine with a ticket.

In [3]:
import math
import re

import numpy as np

customers = R.read_csv(OUT, 'fleet/fx_customer_rates.csv')  # one row per customer in the fleet
machines = R.read_csv(OUT, 'fleet/fx_machine_flags.csv')  # one row per fleet machine with an unplanned ticket
summary = R.read_json(OUT, 'fleet/fx_summary.json')
assert customers.customer.is_unique and machines.serial_id.is_unique and (machines.tickets > 0).all()

fleet_tickets = int(customers.tickets.sum())
fleet_months = customers.months.sum()
fleet_rate = 100 * fleet_tickets / fleet_months
n_fleet_machines = int(customers.machines.sum())  # machines with installed months in the period
print(f'{fleet_tickets:,} unplanned tickets on {n_fleet_machines:,} machines over {fleet_months:,.1f} installed '
      f'machine-months: {fleet_rate:.3f} per 100.')
pd.DataFrame([
    ('fleet/fx_customer_rates.csv, summed over customers', R.num(fleet_tickets), R.num(fleet_months, 1),
     R.num(fleet_rate, 3)),
    ('fleet/fx_machine_flags.csv, summed over machines with a ticket', R.num(machines.tickets.sum()), '', ''),
    ('fleet/fx_summary.json', R.num(summary['fleet']['tickets']), R.num(summary['fleet']['months'], 1),
     R.num(summary['fleet']['rate_per_100'], 3)),
], columns=['source', 'unplanned tickets', 'installed machine-months', 'rate per 100'])

6,750 unplanned tickets on 8,946 machines over 109,621.3 installed machine-months: 6.158 per 100.


,source,unplanned tickets,installed machine-months,rate per 100
0,"fleet/fx_customer_rates.csv, summed over custo...","6,750","109,621.3",6.158
1,"fleet/fx_machine_flags.csv, summed over machin...","6,750",,
2,fleet/fx_summary.json,"6,750","109,621.3",6.158


## The three busiest models: 42% of machine-months and 51% of tickets, partly because they are younger

We rank models by their unplanned tickets in the per-machine table and take each model's installed months from `fleet/fx_model_rates.csv`, which has one row per model. To see how much of their higher rate is age, we compare their tickets with their machine-months at the fleet rate (raw) and with the tickets their machines would have at the fleet's age-band rates (age for age), which the script fits jointly with the model factors over every fleet machine (`expected_age_only`).

In [4]:
models = R.read_csv(OUT, 'fleet/fx_model_rates.csv').set_index('model')  # one row per model
model_tickets = machines.groupby('model').tickets.sum().reindex(models.index, fill_value=0)
assert (model_tickets == models.tickets).all() and math.isclose(models.months.sum(), fleet_months)
top3 = list(model_tickets.sort_index().sort_values(ascending=False, kind='mergesort').index[:3])


def vs_fleet(rows):
    """Shares of the fleet and ratios to the fleet's rate for a set of models."""
    t, mo, e = model_tickets[rows].sum(), models.months[rows].sum(), models.expected_age_only[rows].sum()
    return {'share of machine-months': mo / fleet_months, 'share of tickets': t / fleet_tickets,
            'rate per 100': 100 * t / mo, 'raw ratio': (t / mo) / (fleet_tickets / fleet_months),
            'age-for-age ratio': t / e}


busiest = pd.DataFrame({**{model: vs_fleet([model]) for model in top3}, 'the three together': vs_fleet(top3),
                        'fleet': vs_fleet(list(models.index))}).T
share_months, share_tickets, raw_ratio, age_ratio = busiest.loc[
    'the three together', ['share of machine-months', 'share of tickets', 'raw ratio', 'age-for-age ratio']]
if age_ratio <= 1:
    youth = 'entirely because they are younger'
elif age_ratio < raw_ratio:
    youth = 'partly because they are younger'
else:
    youth = 'not because they are younger'
print(f'Together they run {raw_ratio:.2f} times the fleet rate raw and {age_ratio:.2f} times age for age. The fleet\'s '
      f'age-band rates alone give their machines {raw_ratio / age_ratio:.2f} times the fleet rate, so they run above '
      f'the fleet {youth}.')
busiest.round(3)

Together they run 1.21 times the fleet rate raw and 1.08 times age for age. The fleet's age-band rates alone give their machines 1.12 times the fleet rate, so they run above the fleet partly because they are younger.


,share of machine-months,share of tickets,rate per 100,raw ratio,age-for-age ratio
LVD-Remote-2.0,0.157,0.205,8.008,1.301,1.160
LVRR-WP-2.0,0.141,0.177,7.743,1.258,1.122
MV-2.0,0.125,0.131,6.436,1.045,0.932
the three together,0.423,0.512,7.455,1.211,1.080
fleet,1.000,1.000,6.158,1.000,1.000


## scrubBank 480: 2.6 times the fleet, age for age

A model's age-for-age ratio is its tickets divided by the tickets its machines would have at the fleet's age-band rates. That denominator comes from the script's joint model and age fit over every fleet machine, so we read it from `fleet/fx_model_rates.csv` (`expected_age_only`) and take the tickets from the per-machine table.

In [5]:
sb = 'scrubBank 480'
sb_ratio = model_tickets[sb] / models.expected_age_only[sb]
print(f'{sb}: {model_tickets[sb]} unplanned tickets vs. {models.expected_age_only[sb]:.1f} at the fleet\'s age-band '
      f'rates, {sb_ratio:.3f} times the fleet age for age (fx_model_rates.csv: {models.age_std_ratio[sb]:.3f}, 95% '
      f'interval {models.age_std_lo[sb]:.2f}–{models.age_std_hi[sb]:.2f}; raw: {models.ratio_to_fleet[sb]:.2f} times).')

scrubBank 480: 122 unplanned tickets vs. 47.3 at the fleet's age-band rates, 2.581 times the fleet age for age (fx_model_rates.csv: 2.581, 95% interval 1.75–3.79; raw: 2.66 times).


## Machines under 4 years: about 8 per 100 vs. 2.5 at 10 years or more

Age is the years from install to the midpoint of a machine's installed time in the period, in the script's bands (<1, 1–2, 2–4, 4–6, 6–8, 8–10, and 10+ years). We count each band's tickets in the per-machine table and take its installed months from `fleet/fx_age_rates.csv`, which has one row per band. The bands under 4 years are the bands that run above the fleet rate, so we find them that way and read the 4-year edge from the last one's label.

In [6]:
ages = R.read_csv(OUT, 'fleet/fx_age_rates.csv').set_index('age_band')  # one row per age band, youngest first
band_tickets = machines.groupby('age_band').tickets.sum().reindex(ages.index, fill_value=0)
assert (band_tickets == ages.tickets).all() and math.isclose(ages.months.sum(), fleet_months)
band_rate = 100 * band_tickets / ages.months
above = band_rate > fleet_rate
n_young = int(above.cumprod().sum())  # the leading run of bands above the fleet rate
assert n_young > 0 and not above.iloc[n_young:].any()  # every older band runs below it
young = list(ages.index[:n_young])
young_edge = int(young[-1].split('-')[1])
rate_young = 100 * band_tickets[young].sum() / ages.months[young].sum()
oldest = ages.index[-1]
rate_oldest = band_rate[oldest]
print(f'Under {young_edge} years (bands {", ".join(young)}): {rate_young:.2f} per 100; {oldest}: {rate_oldest:.2f} per '
      f'100; fleet: {fleet_rate:.2f} per 100.')
pd.DataFrame({'machines': ages.machines, 'unplanned tickets': band_tickets,
              'installed machine-months': ages.months.round(0).astype(int), 'rate per 100': band_rate.round(2),
              'vs. the fleet rate': np.where(above, 'above', 'below')})

Under 4 years (bands <1, 1-2, 2-4): 8.05 per 100; 10+: 2.53 per 100; fleet: 6.16 per 100.


,machines,unplanned tickets,installed machine-months,rate per 100,vs. the fleet rate
age_band,,,,,
<1,1497,1071,12708,8.43,above
1-2,866,882,11267,7.83,above
2-4,1624,1676,21129,7.93,above
4-6,1731,1125,22521,5.00,below
6-8,1458,1038,18969,5.47,below
8-10,939,684,12217,5.60,below
10+,831,274,10812,2.53,below


## The 10+ band: 83 of its 164 machines with a ticket are Not Serviceable

Service status is the list's Oct 19th, 2026, snapshot. We count the 10+ band's machines with a ticket by status in the per-machine table and check the total against the band's `machines_with_ticket` in `fleet/fx_age_rates.csv`. The list holds only machines active on Oct 19th, 2026, so removed machines are missing, and we print the unplanned tickets on serials that are not on the list from `fleet/fx_match.csv` as context.

In [7]:
old = machines[machines.age_band == oldest]
n_old = len(old)
n_old_not_serviceable = int((old.service_status == 'Not Serviceable').sum())
assert n_old == ages.machines_with_ticket[oldest]
match = R.read_csv(OUT, 'fleet/fx_match.csv').set_index('measure')['value']
print(f'{n_old_not_serviceable} of the {n_old} machines in the {oldest} band with a ticket are Not Serviceable.')
print(f"Not on the list: {match['unplanned tickets on unlisted serials']:,} unplanned tickets on "
      f"{match['unlisted serials with unplanned tickets']:,} serials, which drop out with their unknown exposure.")
old.service_status.value_counts().rename(f'{oldest} machines with a ticket').rename_axis('service status').to_frame()

83 of the 164 machines in the 10+ band with a ticket are Not Serviceable.
Not on the list: 433 unplanned tickets on 226 serials, which drop out with their unknown exposure.


,10+ machines with a ticket
service status,
Not Serviceable,83
No Coverage,44
Service Contract,37


## 63 customers above their fleets beyond chance: 1,732 tickets vs. 747 expected

A customer's expected tickets are the sum of its machines' fitted expected tickets for their model, age band, and installed months (`expected` in `fleet/fx_customer_rates.csv`). We recompute each customer's one-sided exact Poisson p-value from its tickets and expected tickets, apply Benjamini–Hochberg over the customers with expected tickets, and flag those with q < 0.05 and at least 5 tickets.

In [8]:
def poisson_upper(k, mu):
    """P(X >= k) for X ~ Poisson(mu) with mu > 0, summed over the upper tail so tiny p-values stay accurate."""
    k = int(k)
    if k <= 0:
        return 1.0
    if k <= mu:
        return max(0.0, 1.0 - sum(math.exp(-mu + j * math.log(mu) - math.lgamma(j + 1)) for j in range(k)))
    term = math.exp(-mu + k * math.log(mu) - math.lgamma(k + 1))
    total, j = term, k
    while term > total * 1e-17:
        j += 1
        term *= mu / j
        total += term
    return min(1.0, total)


def upper_p(observed, expected):
    """One-sided Poisson p-values; NaN where nothing is expected (those units are not tested)."""
    return np.array([poisson_upper(o, e) if e > 0 else np.nan for o, e in zip(observed, expected)])


def bh(p):
    """Benjamini–Hochberg q-values over the p-values that are not NaN."""
    p = np.asarray(p, float)
    q = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    n = int(ok.sum())
    order = np.argsort(p[ok], kind='mergesort')
    ranked = p[ok][order] * n / np.arange(1, n + 1)
    q_ok = np.empty(n)
    q_ok[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    q[ok] = q_ok
    return q


p_customer = upper_p(customers.tickets, customers.expected)
q_customer = bh(p_customer)
flagged_c = (customers.tickets >= 5) & (q_customer < 0.05)
assert np.allclose(q_customer, customers.q_poisson, equal_nan=True) and (flagged_c == customers.poisson_flagged).all()
n_flagged = int(flagged_c.sum())
flagged_tickets = int(customers.tickets[flagged_c].sum())
flagged_expected = customers.expected[flagged_c].sum()
print(f'{n_flagged} of the {int((customers.expected > 0).sum()):,} customers with expected tickets are flagged: '
      f'{flagged_tickets:,} tickets vs. {flagged_expected:.1f} expected ({flagged_tickets / fleet_tickets:.1%} of all '
      f'tickets).')
(customers.assign(group=np.where(flagged_c, 'flagged beyond chance', 'not flagged'))
 .groupby('group').agg(customers=('customer', 'size'), tickets=('tickets', 'sum'), expected=('expected', 'sum'))
 .assign(STR=lambda d: d.tickets / d.expected).rename_axis(None).round(2))

63 of the 1,056 customers with expected tickets are flagged: 1,732 tickets vs. 747.4 expected (25.7% of all tickets).


,customers,tickets,expected,STR
flagged beyond chance,63,1732,747.42,2.32
not flagged,1002,5018,6002.58,0.84


## 1 customer outside the normal spread

The overdispersion-adjusted funnel scores each customer on the square-root scale, z_D = (√(O/E) − 1) / √(1/(4E) + τ²), where τ² is the spread between customers, a moment estimate from the unadjusted scores winsorized at their 10th and 90th percentiles. A customer lies outside the normal spread when z_D is above the funnel's 99.8% limit (3.090), and because τ² depends only on each customer's tickets and expected tickets in `fleet/fx_customer_rates.csv`, we recompute it and compare it with `fleet/fx_summary.json`.

In [9]:
def funnel(observed, expected, winsor=0.10):
    """Overdispersion-adjusted funnel scores z_D and tau^2 (Spiegelhalter 2005, square-root scale)."""
    O, E = np.asarray(observed, float), np.asarray(expected, float)
    ok = E > 0
    y, s = np.sqrt(O[ok] / E[ok]), 1 / (2 * np.sqrt(E[ok]))
    z = (y - 1) / s
    zw = np.clip(z, *np.percentile(z, [100 * winsor, 100 * (1 - winsor)]))
    w, n = 1 / s ** 2, int(ok.sum())
    tau2 = max(0.0, (n * np.mean(zw ** 2) - (n - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    zd = np.full(len(O), np.nan)
    zd[ok] = (y - 1) / np.sqrt(s ** 2 + tau2)
    return zd, tau2


z_d, tau2 = funnel(customers.tickets, customers.expected)
z95, z998 = summary['definitions']['z95'], summary['definitions']['z998']
beyond = z_d > z998
n_beyond = int(beyond.sum())
assert np.allclose(z_d, customers.z_d, equal_nan=True) and (beyond == customers.beyond_funnel_998).all()
print(f'tau^2 = {tau2:.5f} (fx_summary.json: {summary["customers"]["tau2"]:.5f}).')
pd.Series({'above the 99.8% limit (outside the normal spread)': n_beyond, 'above the 95% limit': int((z_d > z95).sum()),
           'below the lower 99.8% limit': int((z_d < -z998).sum())}, name='customers').to_frame()

tau^2 = 0.09040 (fx_summary.json: 0.09040).


,customers
above the 99.8% limit (outside the normal spread),1
above the 95% limit,32
below the lower 99.8% limit,0


## The top 10% of customers hold 47% of tickets vs. 39% from their fleets plus chance

We rank the customers with installed months by unplanned tickets and take the top 10%, rounding up to whole customers. The chance benchmark is the median over the script's 1,000 draws of every customer's tickets from a Poisson distribution with its expected tickets, so we read it from `fleet/fx_customer_lorenz.csv` at 10% of customers; the script makes those draws after its permutation tests on machines that the outputs do not list, so we cannot replay them, and we redraw them with our own seed as a cross-check.

In [10]:
active = customers[customers.months > 0]
n_active = len(active)
top_k = math.ceil(0.10 * n_active)
ranked = np.sort(active.tickets.to_numpy())[::-1]
top_share = ranked[:top_k].sum() / ranked.sum()

lorenz = R.read_csv(OUT, 'fleet/fx_customer_lorenz.csv')  # the script's top shares at each 1% of customers
at10 = lorenz[np.isclose(lorenz.customer_share, 0.10)].iloc[0]
chance_share = at10.ticket_share_null_median


def top_shares(x):
    """Share held by the top 10% of units: interpolated at exactly 10% (as the script's Lorenz curve) and by the
    top_k whole units."""
    x = np.sort(np.asarray(x, float))[::-1]
    curve = np.concatenate([[0.0], np.cumsum(x) / x.sum()])
    return float(np.interp(0.10, np.linspace(0, 1, len(x) + 1), curve)), float(curve[top_k])


draws = np.random.default_rng(2026).poisson(active.expected.to_numpy(), size=(summary['n_null'], n_active))
redraw = np.array([top_shares(d) for d in draws])
print(f'The top {top_k} of {n_active:,} customers ({top_k / n_active:.1%}) hold {ranked[:top_k].sum():,} of '
      f'{ranked.sum():,} unplanned tickets: {top_share:.1%}.')
pd.DataFrame({
    'observed': [at10.ticket_share_obs, top_shares(active.tickets)[0], top_share],
    'expected tickets (fleets alone)': [at10.expected_share, *top_shares(active.expected)],
    'fleets plus chance: median': [chance_share, *np.median(redraw, axis=0)],
    'fleets plus chance: 2.5th percentile': [at10.ticket_share_null_p2_5, *np.percentile(redraw, 2.5, axis=0)],
    'fleets plus chance: 97.5th percentile': [at10.ticket_share_null_p97_5, *np.percentile(redraw, 97.5, axis=0)],
}, index=['fx_customer_lorenz.csv (the script, at exactly 10%)', 'recomputed here, at exactly 10% (our draws)',
          f'recomputed here, top {top_k} customers (our draws)']).map(lambda x: R.pct(x, 1))

The top 106 of 1,056 customers (10.0%) hold 3,184 of 6,750 unplanned tickets: 47.2%.


,observed,expected tickets (fleets alone),fleets plus chance: median,fleets plus chance: 2.5th percentile,fleets plus chance: 97.5th percentile
"fx_customer_lorenz.csv (the script, at exactly 10%)",47.1%,37.9%,39.2%,38.1%,40.3%
"recomputed here, at exactly 10% (our draws)",47.1%,37.9%,39.2%,38.2%,40.2%
"recomputed here, top 106 customers (our draws)",47.2%,38.0%,39.3%,38.3%,40.3%


## Customers stay above their fleets: rank correlation 0.46 vs. 0.04 when shuffled

Each half's STR is a customer's tickets in that half divided by its expected tickets refitted within the half (`O_h1`, `E_h1`, `O_h2`, and `E_h2` in `fleet/fx_customer_rates.csv`), and the rank correlation covers the customers with at least 3 expected tickets in each half. The shuffled value is the median over the script's 1,000 permutations that shuffle second-half STRs among customers in the same tenth of second-half expected tickets, so we read it from `fleet/fx_outlier_persistence.csv`, and because these permutations are the first draws from the script's random generator, we replay them with its seed as a cross-check.

In [11]:
persistence = R.read_csv(OUT, 'fleet/fx_outlier_persistence.csv').set_index('statistic')
both_halves = customers[(customers.E_h1 >= 3) & (customers.E_h2 >= 3)].sort_values('customer')  # the script's order
str_h1 = (both_halves.O_h1 / both_halves.E_h1).to_numpy()
str_h2 = (both_halves.O_h2 / both_halves.E_h2).to_numpy()


def spearman(a, b):
    """Spearman rank correlation: the correlation of the ranks, with tied values sharing their average rank."""
    return float(np.corrcoef(pd.Series(a).rank(), pd.Series(b).rank())[0, 1])


rho = spearman(str_h1, str_h2)
row = persistence.loc['spearman_STR_h1_h2']
rho_null = row.perm_null_median
assert row.units == len(both_halves)

# Replay of the script's permutations: same seed, same customer order, same tenths of second-half expected tickets
rng = np.random.default_rng(summary['seed'])
tenth = pd.qcut(both_halves.E_h2.rank(method='first'), 10, labels=False).to_numpy()
replay = []
for _ in range(summary['n_null']):
    idx = np.arange(len(both_halves))
    for g in range(10):
        sel = np.where(tenth == g)[0]
        idx[sel] = rng.permutation(sel)
    replay.append(spearman(str_h1, str_h2[idx]))
print(f'{len(both_halves)} customers with at least 3 expected tickets in each half: rank correlation {rho:.4f}.')
pd.DataFrame({'observed': [row.observed, rho], 'shuffled: median': [rho_null, np.median(replay)],
              'shuffled: 2.5th percentile': [row.perm_null_p2_5, np.percentile(replay, 2.5)],
              'shuffled: 97.5th percentile': [row.perm_null_p97_5, np.percentile(replay, 97.5)]},
             index=['fx_outlier_persistence.csv', 'recomputed here (replay)']).round(4)

311 customers with at least 3 expected tickets in each half: rank correlation 0.4589.


,observed,shuffled: median,shuffled: 2.5th percentile,shuffled: 97.5th percentile
fx_outlier_persistence.csv,0.4589,0.0366,-0.071,0.1468
recomputed here (replay),0.4589,0.0366,-0.071,0.1468


## Machines with 3 or more first-half tickets: 2.6 times their expected tickets in the second half

For the machines with installed time in both halves, we take those with at least 3 first-half tickets, the cut named in the script's statistic in `fleet/fx_outlier_persistence.csv`, and divide their second-half tickets by their second-half expected tickets, refitted within the half, from the per-machine table. We compare the ratio with that table and with the rows by first-half tickets in `fleet/fx_outlier_machine_persistence.csv`.

In [12]:
stat = next(s for s in persistence.index if re.fullmatch(r'second_half_ratio_if_\d+plus_first_half', s))
cut = int(re.search(r'(\d+)plus', stat).group(1))
in_both = machines[(machines.months_h1 > 0) & (machines.months_h2 > 0)]
picked = in_both[in_both.tickets_h1 >= cut]
machine_ratio = picked.tickets_h2.sum() / picked.expected_h2.sum()
assert len(picked) == persistence.units[stat]
print(f'{len(picked)} machines with {cut} or more first-half tickets: {picked.tickets_h2.sum()} second-half tickets vs. '
      f'{picked.expected_h2.sum():.1f} expected, {machine_ratio:.4f} times (fx_outlier_persistence.csv: '
      f'{persistence.observed[stat]:.4f}; first-half counts shuffled within model and age cells give '
      f'{persistence.perm_null_median[stat]:.2f}).')

groups = R.read_csv(OUT, 'fleet/fx_outlier_machine_persistence.csv', dtype={'first_half_tickets': str})
groups = groups.set_index('first_half_tickets').second_half_ratio
rows = {'1': in_both.tickets_h1 == 1, '2': in_both.tickets_h1 == 2, '3': in_both.tickets_h1 == 3,
        '4+': in_both.tickets_h1 >= 4, f'{cut} or more': in_both.tickets_h1 >= cut}
pd.DataFrame([{'machines': int(sel.sum()), 'second-half tickets': int(in_both.tickets_h2[sel].sum()),
               'second-half expected': in_both.expected_h2[sel].sum(),
               'ratio': in_both.tickets_h2[sel].sum() / in_both.expected_h2[sel].sum(),
               'ratio in the script\'s tables': groups.get(name, persistence.observed[stat])}
              for name, sel in rows.items()], index=pd.Index(list(rows), name='first-half tickets')).round(3)

272 machines with 3 or more first-half tickets: 334 second-half tickets vs. 128.8 expected, 2.5932 times (fx_outlier_persistence.csv: 2.5932; first-half counts shuffled within model and age cells give 1.05).


,machines,second-half tickets,second-half expected,ratio,ratio in the script's tables
first-half tickets,,,,,
1,1407,827,613.098,1.349,1.349
2,421,302,187.366,1.612,1.612
3,142,189,66.617,2.837,2.837
4+,130,145,62.180,2.332,2.332
3 or more,272,334,128.797,2.593,2.593


## 18 customers and 36 machines are consistent outliers

A consistent outlier is flagged in at least 10 of the 12 method variants and has a one-sided Poisson p < 0.05 in each half. Each variant refits expected tickets for every fleet machine, including those without a ticket, so we read how many variants flag each customer and machine (`variants_flagged`) from the per-customer and per-machine tables, and we recompute the half p-values from each half's tickets and expected tickets.

In [13]:
min_variants, n_variants = summary['definitions']['min_variants'], summary['definitions']['n_variants']
p1_c, p2_c = upper_p(customers.O_h1, customers.E_h1), upper_p(customers.O_h2, customers.E_h2)
consistent_c = (customers.variants_flagged >= min_variants) & (p1_c < 0.05) & (p2_c < 0.05)
p1_m, p2_m = upper_p(machines.tickets_h1, machines.expected_h1), upper_p(machines.tickets_h2, machines.expected_h2)
consistent_m = (machines.variants_flagged >= min_variants) & (p1_m < 0.05) & (p2_m < 0.05)
assert (consistent_c == customers.consistent_outlier).all() and (consistent_m == machines.consistent_outlier).all()
n_consistent_c, n_consistent_m = int(consistent_c.sum()), int(consistent_m.sum())
pd.DataFrame([{f'flagged in {min_variants}+ of {n_variants} variants': int((t.variants_flagged >= min_variants).sum()),
               'consistent outliers': int(c.sum()), 'their tickets': int(t.tickets[c].sum()),
               'their expected tickets': round(float(t.expected[c].sum()), 1)}
              for t, c in ((customers, consistent_c), (machines, consistent_m))], index=['customers', 'machines'])

,flagged in 10+ of 12 variants,consistent outliers,their tickets,their expected tickets
customers,32,18,737,318.4
machines,58,36,310,31.3


## Of the 106 chronic outliers on IPA's list, 93 stay flagged

docs/05's chronic outliers are the 113 devices labeled `chronic_outlier` in `htm_device_exposure.csv` (notebook 02 rebuilds them). One is on IPA's list when its serial is in the per-machine table, which covers every listed machine that has a ticket or is among the 113, apart from the third-party plan, which holds none of them. A machine stays flagged when it has at least 5 unplanned tickets and a Benjamini–Hochberg q < 0.05 over every fleet machine with installed months, which we recompute from its tickets and expected tickets, counting each machine without a ticket at p = 1.

In [14]:
htm = R.read_csv(OUT, 'htm_device_exposure.csv', usecols=['serial_id', 'class'])  # one row per telemetry device
chronic = set(htm.serial_id[htm['class'] == 'chronic_outlier'])
# Every rate group and age band has tickets, so every fleet machine with installed months has expected tickets above 0
assert (models.tickets[models.own_rate] > 0).all() and models.tickets[~models.own_rate].sum() > 0
assert (ages.tickets > 0).all() and (machines.expected > 0).all()
p_machine = upper_p(machines.tickets, machines.expected)
q_machine = bh(np.concatenate([p_machine, np.ones(n_fleet_machines - len(machines))]))[:len(machines)]
flagged_m = (machines.tickets >= 5) & (q_machine < 0.05)
assert np.allclose(q_machine, machines.q_poisson) and (flagged_m == machines.flagged).all()
on_list = machines.serial_id.isin(chronic)
assert (on_list == machines.in_docs05_113).all()
n_on_list = int(on_list.sum())
n_still_flagged = int((on_list & flagged_m).sum())
print(f'{int(flagged_m.sum())} fleet machines are flagged. Chronic outliers on the third-party plan that the fleet '
      f'leaves out: {summary["docs05_overlap"]["on_list_third_party"]} (fx_summary.json).')
pd.Series({'chronic outliers (docs/05)': len(chronic), "on IPA's list": n_on_list, 'still flagged': n_still_flagged,
           'consistent outliers': int((on_list & consistent_m).sum())}, name='machines').to_frame()

128 fleet machines are flagged. Chronic outliers on the third-party plan that the fleet leaves out: 0 (fx_summary.json).


,machines
chronic outliers (docs/05),113
on IPA's list,106
still flagged,93
consistent outliers,30


## No flagged machine stands out as a separate group

The negative-binomial check lets each model group's ticket rate vary between machines, with a dispersion that the script fits over every fleet machine, including those without a ticket. We read each machine's negative-binomial q-value (`q_nb`) from the per-machine table and count the flagged machines that still have q < 0.05.

In [15]:
n_nb = int((flagged_m & (machines.q_nb < 0.05)).sum())
print(f'{n_nb} of the {int(flagged_m.sum())} flagged machines have a negative-binomial q < 0.05; the smallest q among '
      f'them is {machines.q_nb[flagged_m].min():.3f} (fx_summary.json: {summary["machines"]["flagged_nb"]} machines '
      f'flagged under the negative binomial).')

0 of the 128 flagged machines have a negative-binomial q < 0.05; the smallest q among them is 0.103 (fx_summary.json: 0 machines flagged under the negative binomial).


## Check against the report

In [16]:
R.check([
    ('Fleet rate: unplanned tickets per 100 installed machine-months', '6.2', R.num(fleet_rate, 1)),
    ('Three busiest models by unplanned tickets', 'LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0',
     ', '.join(top3[:-1]) + ', and ' + top3[-1]),
    ('Their share of installed machine-months', '42%', R.pct(share_months)),
    ('Their share of unplanned tickets', '51%', R.pct(share_tickets)),
    ('Their excess: raw vs. age-for-age ratio', 'partly because they are younger', youth),
    ('scrubBank 480 vs. the fleet, age for age (times)', '2.6', R.num(sb_ratio, 1)),
    ('Machines under 4 years: edge of the bands above the fleet rate (years)', '4', R.num(young_edge)),
    ('Rate per 100: machines under 4 years', '8', R.num(rate_young)),
    ('Oldest age band, 10 years or more', '10+', oldest),
    ('Rate per 100: machines 10 years or more', '2.5', R.num(rate_oldest, 1)),
    ('10+ machines with a ticket that are Not Serviceable', '83', R.num(n_old_not_serviceable)),
    ('10+ machines with a ticket', '164', R.num(n_old)),
    ('Customers above their fleets beyond chance', '63', R.num(n_flagged)),
    ('Their unplanned tickets', '1,732', R.num(flagged_tickets)),
    ('Their expected tickets', '747', R.num(flagged_expected)),
    ('Customers outside the normal spread (above the 99.8% funnel limit)', '1', R.num(n_beyond)),
    ('Top group as a share of the customers with installed months', '10%', R.pct(top_k / n_active)),
    ("Top 10% of customers' share of unplanned tickets", '47%', R.pct(top_share)),
    ('Same share from their fleets plus chance (median of 1,000 draws)', '39%', R.pct(chance_share)),
    ('Rank correlation of customer STRs, first vs. second half', '0.46', R.num(rho, 2)),
    ('Same when shuffled (median of 1,000 permutations)', '0.04', R.num(rho_null, 2)),
    ('Machines picked on first-half tickets', '3 or more', f'{cut} or more'),
    ('Their second-half tickets vs. expected (times)', '2.6', R.num(machine_ratio, 1)),
    ('Consistent-outlier customers', '18', R.num(n_consistent_c)),
    ('Consistent-outlier machines', '36', R.num(n_consistent_m)),
    ("docs/05 chronic outliers on IPA's list", '106', R.num(n_on_list)),
    ('Of those, still flagged', '93', R.num(n_still_flagged)),
    ('Flagged machines that stand out under the negative binomial', 'no', 'no' if n_nb == 0 else R.num(n_nb)),
]);

,Finding,Report,Recomputed,Match
0,Fleet rate: unplanned tickets per 100 installe...,6.2,6.2,yes
1,Three busiest models by unplanned tickets,"LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0","LVD-Remote-2.0, LVRR-WP-2.0, and MV-2.0",yes
2,Their share of installed machine-months,42%,42%,yes
3,Their share of unplanned tickets,51%,51%,yes
4,Their excess: raw vs. age-for-age ratio,partly because they are younger,partly because they are younger,yes
5,"scrubBank 480 vs. the fleet, age for age (times)",2.6,2.6,yes
6,Machines under 4 years: edge of the bands abov...,4,4,yes
7,Rate per 100: machines under 4 years,8,8,yes
8,"Oldest age band, 10 years or more",10+,10+,yes
9,Rate per 100: machines 10 years or more,2.5,2.5,yes


All 28 values match the report.
